# Pipecat Smart Turn v3 (аудио-модель)

Семантический VAD: смотрит на **сырое аудио** (не транскрипт) и говорит,
закончил ли человек реплику.

- Модель: Whisper Tiny encoder + linear head, ~8M параметров
- Вход: PCM 16kHz mono, до 8 секунд (берётся хвост реплики)
- Выход: вероятность `complete` (0..1), уже после sigmoid
- 23 языка включая русский
- HF: https://huggingface.co/pipecat-ai/smart-turn-v3


## Установка зависимостей

In [4]:
# !pip install onnxruntime transformers huggingface_hub librosa numpy
# для генерации тестовой речи (опционально): !pip install gtts soundfile

## Загрузка модели

In [ ]:
import numpy as np
import onnxruntime as ort
from transformers import WhisperFeatureExtractor
from huggingface_hub import hf_hub_download

# варианты: smart-turn-v3.2-cpu.onnx / smart-turn-v3.2-gpu.onnx
MODEL_FILE = "smart-turn-v3.2-cpu.onnx"
onnx_path = hf_hub_download("pipecat-ai/smart-turn-v3", MODEL_FILE)

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
feature_extractor = WhisperFeatureExtractor(chunk_length=8)

print("inputs :", [(i.name, i.shape) for i in session.get_inputs()])
print("outputs:", [(o.name, o.shape) for o in session.get_outputs()])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


inputs : [('input_features', ['s6', 80, 800])]
outputs: [('logits', ['s6', 1])]


## Функция предсказания

In [6]:
def predict_endpoint(audio: np.ndarray, sr: int = 16000) -> float:
    import librosa
    audio = audio.astype(np.float32)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    peak = np.max(np.abs(audio)) if audio.size else 1.0
    if peak > 1.0:
        audio = audio / peak
    if len(audio) > 8 * 16000:
        audio = audio[-8 * 16000:]

    feats = feature_extractor(
        audio, sampling_rate=16000, return_tensors="np",
        padding="max_length", max_length=8 * 16000,
        truncation=True, do_normalize=True,
    )
    x = feats.input_features.squeeze(0).astype(np.float32)[None]
    out = session.run(None, {"input_features": x})[0]
    score = float(out[0][0])
    return score

## Смоук-тест (модель запускается)

In [7]:
dummy = (np.random.randn(16000) * 0.1).astype(np.float32)  # 1 сек шума
print("smoke score:", round(predict_endpoint(dummy), 4))

smoke score: 0.9854


## Тест на реальной русской речи (gTTS)

Генерируем несколько фраз через Google TTS.

> TTS читает **любую** фразу с завершающей интонацией, поэтому
> `incomplete` примеры будут получать завышенный score. Для честной оценки
> нужны реальные записи с оборванными репликами. Здесь — только
> проверка что пайплайн работает на живом аудио.

In [8]:
import os
try:
    from gtts import gTTS
    import librosa
    samples = {
        "complete_1":   "Здравствуйте, я хотел бы узнать баланс по своей карте.",
        "complete_2":   "Спасибо большое, до свидания.",
        "incomplete_1": "Мне нужно",
        "incomplete_2": "А скажите пожалуйста какой у меня",
    }
    os.makedirs("/tmp/tts", exist_ok=True)
    print(f"{'sample':<16}{'score':>8}  expected")
    for name, text in samples.items():
        path = f"/tmp/tts/{name}.mp3"
        if not os.path.exists(path):
            gTTS(text, lang="ru").save(path)
        audio, _ = librosa.load(path, sr=16000, mono=True)
        exp = "complete" if name.startswith("complete") else "incomplete"
        print(f"{name:<16}{predict_endpoint(audio):>8.4f}  {exp}")
except Exception as e:
    print("gTTS недоступен, пропускаем:", e)

sample             score  expected
complete_1        0.9425  complete
complete_2        0.9256  complete
incomplete_1      0.8548  incomplete
incomplete_2      0.7858  incomplete


## Замер скорости инференса

In [9]:
import time
audio = (np.random.randn(8 * 16000) * 0.1).astype(np.float32)
predict_endpoint(audio) 
t0 = time.perf_counter()
N = 50
for _ in range(N):
    predict_endpoint(audio)
print(f"avg latency: {(time.perf_counter()-t0)/N*1000:.1f} ms / вызов")

avg latency: 21.9 ms / вызов


## На реальных аудио

In [10]:
import librosa

def predict_file(path: str) -> float:
    audio, sr = librosa.load(path=path, sr=16000, mono=True)
    return predict_endpoint(audio, sr)

In [11]:
import io
import soundfile as sf
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

shard = hf_hub_download(
    "pipecat-ai/smart-turn-data-v3.1-test",
    "data/train-00000-of-00009.parquet",
    repo_type="dataset",
)

table = pq.read_table(shard, columns=["audio", "language", "endpoint_bool"])
rows = table.to_pylist()

ru_samples = []
for r in rows:
    if r["language"] != "rus":
        continue
    audio, sr = sf.read(io.BytesIO(r["audio"]["bytes"]))
    label = 1 if r["endpoint_bool"] else 0
    ru_samples.append((audio, label))

print("ру семплов: ", n_all := len(ru_samples))
print("complete: ", n_comlete := sum(y for _, y in ru_samples), " incomplete: ", n_incomplete := n_all - n_comlete)

ру семплов:  165
complete:  87  incomplete:  78


In [14]:
scores = []
labels = []
for audio, label in ru_samples:
    scores.append(predict_endpoint(audio=audio))
    labels.append(label)

scores = np.array(scores)
labels = np.array(labels)

print("посчитано:", len(scores))
print("score complete (avg):  ", round(scores[labels == 1].mean(), 3))
print("score incomplete (avg):", round(scores[labels == 0].mean(), 3))

посчитано: 165
score complete (avg):   0.944
score incomplete (avg): 0.181


In [18]:
import pandas as pd 

meta_rows = pq.read_table(
    shard, columns=["id", "language", "endpoint_bool", "midfiller", "endfiller"]
).to_pylist()

meta = [
    {"id": r["id"][:8], "midfiller": r["midfiller"], "endfiller": r["endfiller"]}
    for r in meta_rows if r["language"] == "rus"
]

THRESHOLD = 0.5

df = pd.DataFrame(meta)
df["true"] = np.where(labels == 1, "complete", "incomplete")
df["score"] = scores.round(3)
df["pred"] = np.where(scores > THRESHOLD, "complete", "incomplete")
df["correct"] = df["true"] == df["pred"]

print("accuracy:", round(df["correct"].mean(), 3))
df.head(50)

accuracy: 0.897


,id,midfiller,endfiller,true,score,pred,correct
0,0021b64b,True,False,incomplete,0.920,complete,False
1,00b81f4e,False,True,incomplete,0.006,incomplete,True
2,011d6a48,False,False,complete,0.988,complete,True
3,011f15aa,True,True,incomplete,0.901,complete,False
4,012107f8,True,False,incomplete,0.005,incomplete,True
5,012d3f9d,True,True,incomplete,0.006,incomplete,True
6,0138b785,False,False,complete,0.931,complete,True
7,013e377b,True,False,complete,0.974,complete,True
8,0173da41,True,True,incomplete,0.005,incomplete,True
9,0189c078,True,True,incomplete,0.018,incomplete,True


### Послушать реальные семплы (меняем индекс у listen)

In [22]:
from IPython.display import Audio, display

def listen(index: int) -> None:
    audio, label = ru_samples[index]
    print(f"#{index} | true={'complete' if label else 'incomplete'} | score={round(scores[index], 3)}")
    display(Audio(audio, rate=16000))

listen(49)

#49 | true=complete | score=0.986
